> ⚠️ **Sobre este notebook.** As saídas foram removidas por conterem os nomes dos voluntários
> participantes, hoje pseudonimizados como `A1`–`A3` (Grupo A) e `B1`–`B3` (Grupo B) na estrutura
> de diretórios.
>
> O notebook, como está, **não executa com as versões atuais** de matplotlib e seaborn: a célula de
> importação depende de `matplotlib.cm.get_cmap`, removido nas versões recentes. Ele é mantido como
> registro histórico da análise exploratória inicial.
>
> As figuras de distribuição Likert publicadas na dissertação **não vêm mais daqui**: são geradas
> por `gerar_figuras_dissertacao.py`, que lê os mesmos dados brutos e não depende de bibliotecas
> obsoletas. As análises estatísticas estão em `evaluation_statistics.ipynb`, que executa sem erros.


## Notebook para processar dados que foram avaliados pelos voluntarios

In [ ]:
import pandas as pd
import json
import glob
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch
import colorsys
sns.set_theme()

In [ ]:
infos_df = []
for arq in glob.iglob("./evaluations/**/*.csv", recursive=True):
    split_name = arq.split("/")
    person = split_name[1]
    subject = split_name[2]
    id_activity = split_name[3]
    id_activity_generated = split_name[4]

    parameters = arq.replace("form.csv", "parameters.json")

    with open(parameters, "r") as f:
        params = json.load(f)
    
    df = pd.read_csv(arq)

    mapping_tasks = {
        "observe image, write answer": "Observar imagem e escrever resposta",
        "observe image, draw answer": "Observar imagem e desenhar resposta",
        "observe images, write answer": "Observar imagens e escrever resposta",
        "observe images, link items": "Observar imagens e relacionar objetos",
        "observe images, link objects": "Observar imagens e relacionar objetos",
    }

    for index, row in df.iterrows():
        info = {

            "person": person,
            "subject": subject,
            "id_activity": id_activity,
            "index_question": f"P{index+1}",
            "learning_goals": params["learning goals"],
            "task": mapping_tasks[params["task"]],
            "temperature": params["temperature"],
            "theme": params["theme"],
            "question": row["Pergunta"],
            "answer": row["Resposta"],
            "model": params["model"],
            "id_generated": id_activity_generated,
        }
    
        infos_df.append(info)

final_df = pd.DataFrame(infos_df)


In [ ]:
display(final_df.head(20))

In [ ]:
# def plot_likert_distribution(final_df, model, learning_goal):

#     import numpy as np
#     import matplotlib.pyplot as plt
#     from matplotlib.patches import Patch

#     # =========================
#     # Filtragem dos dados
#     # =========================
#     df = final_df[
#         (final_df["model"] == model) &
#         (final_df["learning_goals"] == learning_goal) &
#         (final_df["temperature"].isin([0.5, 0.7]))
#     ]

#     tasks = [
#         'Observar imagem e escrever resposta',
#         'Observar imagem e desenhar resposta',
#         'Observar imagens e escrever resposta',
#         'Observar imagens e relacionar objetos'
#     ]

#     temperatures = [0.5, 0.7]
#     temp_hatch = {0.5: '', 0.7: '//'}
#     order = [f'P{i}' for i in range(1, 11)]

#     # =========================
#     # Paleta Likert fixa
#     # 1 vermelho → 5 verde
#     # =========================
#     likert_levels = [1, 2, 3, 4, 5]

#     colors = {
#         1: '#8c2d2d',   # vermelho escuro opaco
#         2: '#c07a7a',   # vermelho claro opaco
#         3: '#f2f2f2',   # branco acinzentado
#         4: '#8fbc8f',   # verde claro opaco
#         5: '#3f6f4f'    # verde escuro opaco
#     }

#     # =========================
#     # Figura com subplots
#     # =========================
#     fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharey=True)
#     axes = axes.flatten()

#     bar_width = 0.35
#     x = np.arange(len(order))

#     for ax, task in zip(axes, tasks):
#         for i, temp in enumerate(temperatures):

#             subset = df[
#                 (df['task'] == task) &
#                 (df['temperature'] == temp)
#             ]

#             df_counts = (
#                 subset
#                 .groupby(['index_question', 'answer'])
#                 .size()
#                 .reset_index(name='count')
#             )

#             df_counts['percent'] = (
#                 df_counts
#                 .groupby('index_question')['count']
#                 .transform(lambda x: x / x.sum())
#             )

#             df_pivot = (
#                 df_counts
#                 .pivot(index='index_question', columns='answer', values='percent')
#                 .fillna(0)
#             )

#             # Garante ordem Likert 1 → 5
#             df_pivot = df_pivot.reindex(columns=likert_levels, fill_value=0)

#             # Ordem P1 → P10
#             ordered_index = [q for q in order if q in df_pivot.index]
#             df_pivot = df_pivot.loc[ordered_index]

#             bottom = np.zeros(len(df_pivot))

#             for answer in df_pivot.columns:
#                 ax.bar(
#                     x[:len(df_pivot)] + (i - 0.5) * bar_width,
#                     df_pivot[answer],
#                     bar_width,
#                     bottom=bottom,
#                     color=colors[answer],
#                     edgecolor='black',
#                     hatch=temp_hatch[temp]
#                 )
#                 bottom += df_pivot[answer].values

#             ax.set_ylim(-0.05, 1.05)

#         ax.set_title(task)
#         ax.set_xticks(x)
#         ax.set_xticklabels(order)
#         ax.set_xlabel('Pergunta')

#     axes[0].set_ylabel('Proporção de respostas')

#     # =========================
#     # Legenda Likert
#     # =========================
#     likert_handles = [
#         Patch(facecolor=colors[lvl], edgecolor='black', label=str(lvl))
#         for lvl in likert_levels
#     ]

#     fig.legend(
#         likert_handles,
#         [str(lvl) for lvl in likert_levels],
#         title='Escala Likert',
#         loc='lower center',
#         ncol=5,
#         frameon=False
#     )

#     # =========================
#     # Legenda Temperatura
#     # =========================
#     temp_handles = [
#         Patch(facecolor='white', edgecolor='black', hatch='', label='Temperatura 0.5'),
#         Patch(facecolor='white', edgecolor='black', hatch='//', label='Temperatura 0.7')
#     ]

#     fig.legend(
#         temp_handles,
#         ['Temperatura 0.5', 'Temperatura 0.7'],
#         loc='upper right',
#         frameon=False
#     )

#     fig.suptitle(
#         f'Distribuição das respostas Likert por pergunta, tipo de tarefa e temperatura\n{learning_goal.capitalize()}',
#         fontsize=14
#     )

#     plt.tight_layout(rect=[0, 0.08, 1, 0.95])

#     name_save = f"likert_distribution_gpt41_{learning_goal}"
#     plt.savefig(f'./artifacts/{name_save}.png', dpi=300)
#     plt.show()

#     # =========================
#     # Estatísticas descritivas
#     # =========================
#     stats_long = (
#         df
#         .groupby(['task', 'index_question', 'temperature'])['answer']
#         .agg(
#             N='count',
#             Média='mean',
#             Mediana='median',
#             Desvio_Padrão='std'
#         )
#         .reset_index()
#     )

#     stats_long['question_num'] = (
#         stats_long['index_question']
#         .str.extract('(\d+)')
#         .astype(int)
#     )

#     # =========================
#     # Pivot
#     # =========================
#     stats_wide = (
#         stats_long
#         .pivot_table(
#             index=['task', 'index_question', 'question_num'],
#             columns='temperature',
#             values=['N', 'Média', 'Mediana', 'Desvio_Padrão']
#         )
#     )

#     stats_wide.columns = [
#         f'{metric}_T{temp}'
#         for metric, temp in stats_wide.columns
#     ]

#     stats_final = (
#         stats_wide
#         .reset_index()
#         .sort_values(['task', 'question_num'])
#         .drop(columns='question_num')
#         .reset_index(drop=True)
#     )

#     # =========================
#     # Limita casas decimais (máx. 2)
#     # =========================
#     numeric_cols = stats_final.select_dtypes(include='number').columns
#     stats_final[numeric_cols] = stats_final[numeric_cols].round(2)

#     stats_final.to_csv(
#         f'./artifacts/{name_save}.csv',
#         index=False,
#         float_format='%.2f'
#     )

#     display(stats_final)

def plot_likert_distribution(final_df, model, learning_goal):

    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    # =========================
    # Data filtering
    # =========================
    df = final_df[
        (final_df["model"] == model) &
        (final_df["learning_goals"] == learning_goal) &
        (final_df["temperature"].isin([0.5, 0.7]))
    ]

    # =========================
    # Learning goal translation (only for title)
    # =========================
    learning_goal_translation = {
        "adição": "Addition",
        "subtração": "Subtraction",
        "multiplicação": "Multiplication",
        "divisão": "Division"
    }

    learning_goal_en = learning_goal_translation.get(learning_goal, learning_goal)

    # =========================
    # Tasks in Portuguese (used for filtering)
    # =========================
    tasks_pt = [
        'Observar imagem e escrever resposta',
        'Observar imagem e desenhar resposta',
        'Observar imagens e escrever resposta',
        'Observar imagens e relacionar objetos'
    ]

    # Translation only for display
    task_translation = {
        'Observar imagem e escrever resposta': 'Observe image and write answer',
        'Observar imagem e desenhar resposta': 'Observe image and draw answer',
        'Observar imagens e escrever resposta': 'Observe images and write answer',
        'Observar imagens e relacionar objetos': 'Observe images and match objects'
    }

    temperatures = [0.5, 0.7]
    temp_hatch = {0.5: '', 0.7: '//'}
    order = [f'P{i}' for i in range(1, 11)]

    likert_levels = [1, 2, 3, 4, 5]

    colors = {
        1: '#8c2d2d',
        2: '#c07a7a',
        3: '#f2f2f2',
        4: '#8fbc8f',
        5: '#3f6f4f'
    }

    # =========================
    # Figure
    # =========================
    fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharey=True)
    axes = axes.flatten()

    bar_width = 0.35
    x = np.arange(len(order))

    for ax, task_pt in zip(axes, tasks_pt):

        task_en = task_translation[task_pt]

        for i, temp in enumerate(temperatures):

            subset = df[
                (df['task'] == task_pt) &
                (df['temperature'] == temp)
            ]

            if subset.empty:
                continue

            df_counts = (
                subset
                .groupby(['index_question', 'answer'])
                .size()
                .reset_index(name='count')
            )

            df_counts['percent'] = (
                df_counts
                .groupby('index_question')['count']
                .transform(lambda x: x / x.sum())
            )

            df_pivot = (
                df_counts
                .pivot(index='index_question', columns='answer', values='percent')
                .fillna(0)
            )

            df_pivot = df_pivot.reindex(columns=likert_levels, fill_value=0)

            ordered_index = [q for q in order if q in df_pivot.index]
            df_pivot = df_pivot.loc[ordered_index]

            bottom = np.zeros(len(df_pivot))

            for answer in df_pivot.columns:
                ax.bar(
                    x[:len(df_pivot)] + (i - 0.5) * bar_width,
                    df_pivot[answer],
                    bar_width,
                    bottom=bottom,
                    color=colors[answer],
                    edgecolor='black',
                    hatch=temp_hatch[temp]
                )
                bottom += df_pivot[answer].values

            ax.set_ylim(-0.05, 1.05)

        ax.set_title(task_en)
        ax.set_xticks(x)
        ax.set_xticklabels(order)
        ax.set_xlabel('Question')

    axes[0].set_ylabel('Response Proportion')

    # =========================
    # Likert legend
    # =========================
    likert_handles = [
        Patch(facecolor=colors[lvl], edgecolor='black', label=str(lvl))
        for lvl in likert_levels
    ]

    fig.legend(
        likert_handles,
        [str(lvl) for lvl in likert_levels],
        title='Likert Scale',
        loc='lower center',
        ncol=5,
        frameon=False
    )

    # =========================
    # Temperature legend
    # =========================
    temp_handles = [
        Patch(facecolor='white', edgecolor='black', hatch='', label='Temperature 0.5'),
        Patch(facecolor='white', edgecolor='black', hatch='//', label='Temperature 0.7')
    ]

    fig.legend(
        temp_handles,
        ['Temperature 0.5', 'Temperature 0.7'],
        loc='upper right',
        frameon=False
    )

    fig.suptitle(
        f'Likert Response Distribution by Question, Task Type, and Temperature\n{learning_goal_en}',
        fontsize=14
    )

    plt.tight_layout(rect=[0, 0.08, 1, 0.95])

    name_save = f"likert_distribution_gpt41_{learning_goal_en.lower()}"
    plt.savefig(f'./artifacts/{name_save}.png', dpi=300)
    plt.show()

### Atividades de adição geradas pelo GPT

In [ ]:
plot_likert_distribution(final_df, model="ft:gpt-4.1-2025-04-14:ufcg::CikEQZns", learning_goal="adição")

### Atividades de adição extraídas dos livros

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


# def plot_likert_distribution_books_activities(final_df, model, learning_goal, temperature):

#     # =========================
#     # Filtragem dos dados
#     # =========================
#     additon_model = final_df[
#         (final_df["model"] == model) &
#         (final_df["learning_goals"] == learning_goal) &
#         (final_df["temperature"] == temperature)
#     ]

#     print(additon_model.shape)

#     tasks = [
#         'Observar imagem e escrever resposta',
#         'Observar imagem e desenhar resposta',
#         'Observar imagens e escrever resposta',
#         'Observar imagens e relacionar objetos'
#     ]

#     order = [f'P{i}' for i in range(1, 11)]

#     # =========================
#     # Figura
#     # =========================
#     fig, axes = plt.subplots(
#         nrows=2,
#         ncols=2,
#         figsize=(14, 8),
#         sharey=True
#     )

#     axes = axes.flatten()

#     # =========================
#     # Paleta Likert opaca (vermelho → branco → verde)
#     # =========================
#     likert_levels = [1, 2, 3, 4, 5]

#     likert_color_map = {
#         1: '#8c2d2d',   # vermelho escuro opaco
#         2: '#c07a7a',   # vermelho claro opaco
#         3: '#f2f2f2',   # branco acinzentado
#         4: '#8fbc8f',   # verde claro opaco
#         5: '#3f6f4f'    # verde escuro opaco
#     }

#     # =========================
#     # Loop por tarefa
#     # =========================
#     for ax, task in zip(axes, tasks):

#         subset = additon_model[additon_model['task'] == task]
#         print(subset.shape)

#         df_counts = (
#             subset
#             .groupby(['index_question', 'answer'])
#             .size()
#             .reset_index(name='count')
#         )

#         df_counts['percent'] = (
#             df_counts
#             .groupby('index_question')['count']
#             .transform(lambda x: x / x.sum())
#         )

#         df_pivot = (
#             df_counts
#             .pivot(
#                 index='index_question',
#                 columns='answer',
#                 values='percent'
#             )
#             .fillna(0)
#         )

#         # garante ordem Likert completa
#         df_pivot = df_pivot.reindex(columns=likert_levels, fill_value=0)

#         # ordem correta P1 → P10
#         ordered_index = [q for q in order if q in df_pivot.index]
#         df_pivot = df_pivot.loc[ordered_index]

#         bottom = pd.Series(0, index=df_pivot.index)

#         for answer in df_pivot.columns:
#             ax.bar(
#                 df_pivot.index,
#                 df_pivot[answer],
#                 bottom=bottom,
#                 color=likert_color_map[answer],
#                 edgecolor='black',
#                 linewidth=0.5
#             )
#             bottom += df_pivot[answer]

#         ax.set_title(task)
#         ax.set_xlabel('Pergunta')
#         ax.set_ylim(-0.05, 1.05)

#     axes[0].set_ylabel('Proporção de respostas')

#     # =========================
#     # Legenda Likert
#     # =========================
#     likert_handles = [
#         Patch(
#             facecolor=likert_color_map[lvl],
#             edgecolor='black',
#             label=str(lvl)
#         )
#         for lvl in likert_levels
#     ]

#     fig.legend(
#         likert_handles,
#         [str(lvl) for lvl in likert_levels],
#         title='Escala Likert',
#         loc='lower center',
#         ncol=len(likert_levels),
#         frameon=False
#     )

#     # =========================
#     # Título geral
#     # =========================
#     fig.suptitle(
#         'Distribuição das respostas Likert por pergunta e tipo de tarefa',
#         fontsize=14
#     )

#     plt.tight_layout(rect=[0, 0.08, 1, 0.95])
#     plt.savefig(
#         './artifacts/likert_distribution_books_activities.png',
#         dpi=300
#     )
#     plt.show()

#     # =========================
#     # Estatísticas descritivas
#     # =========================
#     stats_task_question = (
#         additon_model
#         .groupby(['task', 'index_question'])['answer']
#         .agg(
#             N='count',
#             Média='mean',
#             Mediana='median',
#             Desvio_Padrão='std'
#         )
#         .reset_index()
#     )

#     stats_task_question['question_num'] = (
#         stats_task_question['index_question']
#         .str.extract(r'(\d+)')
#         .astype(int)
#     )

#     stats_task = (
#         stats_task_question
#         .sort_values(['task', 'question_num'])
#         .drop(columns='question_num')
#         .reset_index(drop=True)
#     )

#     # limita casas decimais
#     for col in ['Média', 'Mediana', 'Desvio_Padrão']:
#         stats_task[col] = stats_task[col].round(2)

#     stats_task.to_csv(
#         './artifacts/likert_distribution_books_activities.csv',
#         index=False
#     )

#     display(stats_task)


def plot_likert_distribution_books_activities(final_df, model, learning_goal, temperature):

    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    # =========================
    # Data filtering
    # =========================
    filtered_df = final_df[
        (final_df["model"] == model) &
        (final_df["learning_goals"] == learning_goal) &
        (final_df["temperature"] == temperature)
    ]

    print(filtered_df.shape)

    # =========================
    # Learning goal translation (display only)
    # =========================
    learning_goal_translation = {
        "adição": "Addition",
        "subtração": "Subtraction",
        "multiplicação": "Multiplication",
        "divisão": "Division"
    }

    learning_goal_en = learning_goal_translation.get(learning_goal, learning_goal)

    # =========================
    # Tasks in Portuguese (for filtering)
    # =========================
    tasks_pt = [
        'Observar imagem e escrever resposta',
        'Observar imagem e desenhar resposta',
        'Observar imagens e escrever resposta',
        'Observar imagens e relacionar objetos'
    ]

    # Translation for display only
    task_translation = {
        'Observar imagem e escrever resposta': 'Observe image and write answer',
        'Observar imagem e desenhar resposta': 'Observe image and draw answer',
        'Observar imagens e escrever resposta': 'Observe images and write answer',
        'Observar imagens e relacionar objetos': 'Observe images and match objects'
    }

    order = [f'P{i}' for i in range(1, 11)]

    # =========================
    # Figure
    # =========================
    fig, axes = plt.subplots(
        nrows=2,
        ncols=2,
        figsize=(14, 8),
        sharey=True
    )

    axes = axes.flatten()

    # =========================
    # Likert color palette
    # =========================
    likert_levels = [1, 2, 3, 4, 5]

    likert_color_map = {
        1: '#8c2d2d',
        2: '#c07a7a',
        3: '#f2f2f2',
        4: '#8fbc8f',
        5: '#3f6f4f'
    }

    # =========================
    # Loop per task
    # =========================
    for ax, task_pt in zip(axes, tasks_pt):

        task_en = task_translation[task_pt]

        subset = filtered_df[filtered_df['task'] == task_pt]
        print(subset.shape)

        if subset.empty:
            ax.set_title(task_en)
            ax.set_xlabel('Question')
            ax.set_ylim(-0.05, 1.05)
            continue

        df_counts = (
            subset
            .groupby(['index_question', 'answer'])
            .size()
            .reset_index(name='count')
        )

        df_counts['percent'] = (
            df_counts
            .groupby('index_question')['count']
            .transform(lambda x: x / x.sum())
        )

        df_pivot = (
            df_counts
            .pivot(
                index='index_question',
                columns='answer',
                values='percent'
            )
            .fillna(0)
        )

        df_pivot = df_pivot.reindex(columns=likert_levels, fill_value=0)

        ordered_index = [q for q in order if q in df_pivot.index]
        df_pivot = df_pivot.loc[ordered_index]

        bottom = pd.Series(0, index=df_pivot.index)

        for answer in df_pivot.columns:
            ax.bar(
                df_pivot.index,
                df_pivot[answer],
                bottom=bottom,
                color=likert_color_map[answer],
                edgecolor='black',
                linewidth=0.5
            )
            bottom += df_pivot[answer]

        ax.set_title(task_en)
        ax.set_xlabel('Question')
        ax.set_ylim(-0.05, 1.05)

    axes[0].set_ylabel('Response Proportion')

    # =========================
    # Likert legend
    # =========================
    likert_handles = [
        Patch(
            facecolor=likert_color_map[lvl],
            edgecolor='black',
            label=str(lvl)
        )
        for lvl in likert_levels
    ]

    fig.legend(
        likert_handles,
        [str(lvl) for lvl in likert_levels],
        title='Likert Scale',
        loc='lower center',
        ncol=len(likert_levels),
        frameon=False
    )

    # =========================
    # Global title
    # =========================
    fig.suptitle(
        f'Likert Response Distribution by Question and Task Type\n{learning_goal_en}',
        fontsize=14
    )

    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    plt.savefig(
        './artifacts/likert_distribution_books_activities.png',
        dpi=300
    )
    plt.show()

    # =========================
    # Descriptive statistics
    # =========================
    stats_task_question = (
        filtered_df
        .groupby(['task', 'index_question'])['answer']
        .agg(
            N='count',
            Mean='mean',
            Median='median',
            Std_Dev='std'
        )
        .reset_index()
    )

    stats_task_question['question_num'] = (
        stats_task_question['index_question']
        .str.extract(r'(\d+)')
        .astype(int)
    )

    stats_task = (
        stats_task_question
        .sort_values(['task', 'question_num'])
        .drop(columns='question_num')
        .reset_index(drop=True)
    )

    for col in ['Mean', 'Median', 'Std_Dev']:
        stats_task[col] = stats_task[col].round(2)

    stats_task.to_csv(
        './artifacts/likert_distribution_books_activities.csv',
        index=False
    )

    display(stats_task)


In [ ]:
plot_likert_distribution_books_activities(final_df, model="N/A", learning_goal="adição", temperature="N/A")

### Atividades de subtração geradas pelo GPT

In [ ]:
plot_likert_distribution(final_df, model="ft:gpt-4.1-2025-04-14:ufcg::CikEQZns", learning_goal="subtração")

### Atividades de multiplicação geradas pelo GPT

In [ ]:
plot_likert_distribution(final_df, model="ft:gpt-4.1-2025-04-14:ufcg::CikEQZns", learning_goal="multiplicação")

### Atividades de divisão geradas pelo GPT

In [ ]:
plot_likert_distribution(final_df, model="ft:gpt-4.1-2025-04-14:ufcg::CikEQZns", learning_goal="divisão")

### Comparação entre os casos

In [ ]:
final_df

#### Para todos

In [ ]:
# =========================
# Preparação dos dados
# =========================
df = final_df.copy()

# Regra semântica:
# model == 'N/A' → temperature == 'N/A'
df.loc[df['model'] == 'N/A', 'temperature'] = 'N/A'

df['temperature'] = df['temperature'].astype(str)

# =========================
# Média por pergunta
# =========================
mean_per_question = (
    df
    .groupby(['id_activity', 'model', 'temperature', 'index_question'])['answer']
    .mean()
    .reset_index()
)

mean_per_question['question_num'] = (
    mean_per_question['index_question']
    .str.extract(r'(\d+)')
    .astype(int)
)

mean_per_question = mean_per_question.sort_values('question_num')

# =========================
# Paleta semântica por atividade
# =========================
activities = mean_per_question['id_activity'].unique()
base_cmap = plt.cm.tab10

activity_color_map = {
    activity: base_cmap(i % 10)
    for i, activity in enumerate(activities)
}

# =========================
# Gráfico único
# =========================
fig, ax = plt.subplots(figsize=(13, 8))

for (activity, model, temp), group in mean_per_question.groupby(
    ['id_activity', 'model', 'temperature']
):

    # Atividades extraídas
    if model == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns":
        label = f'{activity} | GPT-4.1 Treinado | Temp: {temp}'
        color = activity_color_map[activity]
        linewidth = 2.8
        alpha = 0.65
    else:
        label = 'Atividades extraídas'
        color = 'black'
        linewidth = 3.2
        alpha = 1.0  # sem transparência

    # Estilo de linha por temperatura
    if temp == '0.5':
        linestyle = '-'
    elif temp == '0.7':
        linestyle = ':'
    else:  # N/A
        linestyle = '--'

    ax.plot(
        group['question_num'],
        group['answer'],
        marker='o',
        linewidth=linewidth,
        linestyle=linestyle,
        color=color,
        alpha=alpha,
        label=label
    )

# =========================
# Eixos e layout
# =========================
ax.set_xticks(range(1, 11))
ax.set_xticklabels([f'P{i}' for i in range(1, 11)])

ax.set_ylim(1, 5.2)
ax.set_ylabel('Média das respostas (Likert)')
ax.set_xlabel('Pergunta')

ax.set_title(
    'Média das avaliações Likert por pergunta\n'
    '(agrupado por atividade, modelo e temperatura)',
    fontsize=13
)

ax.grid(axis='y', linestyle='--', alpha=0.4)

# =========================
# Legenda abaixo do gráfico
# =========================
handles, labels = ax.get_legend_handles_labels()

ax.legend(
    handles,
    labels,
    title='Configuração experimental',
    loc='upper center',
    bbox_to_anchor=(0.5, -0.28),
    ncol=2,
    frameon=False
)

plt.tight_layout()
plt.savefig(f'./artifacts/mean_likert_per_question_all_activities.png', dpi=300)
plt.show()

In [ ]:
# # =========================
# # Preparação dos dados
# # =========================
# df = final_df.copy()

# df.loc[df['model'] == 'N/A', 'temperature'] = 'N/A'
# df['temperature'] = df['temperature'].astype(str)

# df['source'] = df['model'].apply(
#     lambda x: 'Modelo Treinado' if x == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns"
#     else 'Atividades extraídas'
# )

# df['question_num'] = df['index_question'].str.extract('(\d+)').astype(int)

# df['config'] = (
#     df['id_activity'] + ' | ' +
#     df['source'] + ' | T=' + df['temperature']
# )

# # =========================
# # Paleta sóbria por assunto (CORRIGIDA)
# # =========================
# activities = df['id_activity'].unique()

# base_colors = sns.color_palette("tab10", n_colors=len(activities))
# base_colors = [sns.desaturate(c, 0.30) for c in base_colors]
# base_colors = [tuple(np.clip(np.array(c) * 0.85, 0, 1)) for c in base_colors]

# activity_color_map = {
#     act: base_colors[i]
#     for i, act in enumerate(activities)
# }

# def vary_lightness(rgb, factors):
#     return [tuple(np.clip(np.array(rgb) * f, 0, 1)) for f in factors]

# palette = {}

# RED_CARO = (0.60, 0.10, 0.10)   # RGB normalizado

# for act in activities:
#     configs_act = [c for c in df['config'].unique() if c.startswith(act + ' | ')]

#     # variações muito sutis de luminosidade
#     factors = np.linspace(0.85, 1.15, len(configs_act))
#     variations = vary_lightness(activity_color_map[act], factors)

#     for c, v in zip(configs_act, variations):
#         _, source, _ = c.split(' | ')
#         if source == 'Atividades extraídas':
#             palette[c] = RED_CARO   # vermelho caro
#         else:
#             palette[c] = v

# # =========================
# # Gráfico Boxplot
# # =========================
# plt.figure(figsize=(15, 8))

# ax = sns.boxplot(
#     data=df,
#     x='question_num',
#     y='answer',
#     hue='config',
#     palette=palette,
#     showfliers=True
# )

# plt.ylim(0.8, 5.2)
# plt.xlabel('Pergunta')
# plt.ylabel('Distribuição das respostas (Likert)')
# plt.title('Distribuição das respostas Likert por pergunta,\nassunto, modelo e temperatura')

# plt.grid(axis='y', linestyle='--', alpha=0.35)

# # =========================
# # Rótulos do eixo X: P1...Pn
# # =========================
# current_labels = [t.get_text() for t in ax.get_xticklabels()]
# new_labels = [f'P{label}' for label in current_labels]
# ax.set_xticklabels(new_labels)

# # =========================
# # Legenda limpa
# # =========================
# handles, labels_leg = ax.get_legend_handles_labels()

# new_labels = []
# for label in labels_leg:
#     act, source, temp = label.split(' | ')
#     if source == 'Atividades extraídas':
#         new_labels.append('Atividades extraídas')
#     else:
#         new_labels.append(f'{act} | {source} | {temp}')

# ax.legend(
#     handles,
#     new_labels,
#     title='Configuração experimental',
#     loc='upper center',
#     bbox_to_anchor=(0.5, -0.25),
#     ncol=3,
#     frameon=False
# )

# plt.tight_layout()
# plt.savefig('./artifacts/boxplot_likert_por_assunto_modelo_temperatura.png', dpi=300)
# plt.show()

# =========================
# Data preparation
# =========================
df = final_df.copy()

df.loc[df['model'] == 'N/A', 'temperature'] = 'N/A'
df['temperature'] = df['temperature'].astype(str)

df['source'] = df['model'].apply(
    lambda x: 'Trained Model' if x == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns"
    else 'Extracted Activities'
)

df['question_num'] = df['index_question'].str.extract('(\d+)').astype(int)

df['config'] = (
    df['id_activity'] + ' | ' +
    df['source'] + ' | T=' + df['temperature']
)

# =========================
# Activity translation (display only)
# =========================
activity_translation = {
    "adição": "Addition",
    "subtração": "Subtraction",
    "multiplicação": "Multiplication",
    "divisão": "Division"
}

# =========================
# Custom hue ordering
# Extracted block stays in the middle
# =========================
activity_order = ["adição", "subtração", "multiplicação", "divisão"]

def hue_sort_key(label):
    act, source, temp = label.split(' | ')

    # Extracted block goes to the middle (group = 1)
    if source == 'Extracted Activities':
        group = 1
        temp_order = 0
    else:
        # Before extracted: addition and subtraction
        if act in ["adição", "subtração"]:
            group = 0
        else:
            # After extracted: multiplication and division
            group = 2

        # Temperature ordering inside group
        if 'T=0.5' in temp:
            temp_order = 0
        elif 'T=0.7' in temp:
            temp_order = 1
        else:
            temp_order = 2

    return (group, activity_order.index(act), temp_order)

hue_order = sorted(df['config'].unique(), key=hue_sort_key)

# =========================
# Boxplot
# =========================
plt.figure(figsize=(15, 8))

ax = sns.boxplot(
    data=df,
    x='question_num',
    y='answer',
    hue='config',
    palette=palette,
    showfliers=True,
    hue_order=hue_order
)

plt.ylim(0.8, 5.2)
plt.xlabel('Question')
plt.ylabel('Response Distribution (Likert)')
plt.title('Likert Response Distribution by Question,\nActivity, Model, and Temperature')

plt.grid(axis='y', linestyle='--', alpha=0.35)

# =========================
# X-axis labels
# =========================
current_labels = [t.get_text() for t in ax.get_xticklabels()]
new_labels = [f'P{label}' for label in current_labels]
ax.set_xticklabels(new_labels)

# =========================
# Clean legend
# =========================
handles, labels_leg = ax.get_legend_handles_labels()

new_labels = []
for label in labels_leg:
    act, source, temp = label.split(' | ')
    act_display = activity_translation.get(act, act)

    if source == 'Extracted Activities':
        new_labels.append('Extracted Activities')
    else:
        new_labels.append(f'{act_display} | {source} | {temp}')

ax.legend(
    handles,
    new_labels,
    title='Experimental Configuration',
    loc='upper center',
    bbox_to_anchor=(0.5, -0.25),
    ncol=3,
    frameon=False
)

plt.tight_layout()
plt.savefig('./artifacts/boxplot_likert_by_activity_model_temperature.png', dpi=300)
plt.show()


#### Para adição

In [ ]:
# =========================
# Preparação dos dados
# =========================
df = final_df.copy()

df = df[df['id_activity'] == "adição"]

# Regra semântica:
# model == 'N/A' → temperature == 'N/A'
df.loc[df['model'] == 'N/A', 'temperature'] = 'N/A'

df['temperature'] = df['temperature'].astype(str)

# =========================
# Média por pergunta
# =========================
mean_per_question = (
    df
    .groupby(['id_activity', 'model', 'temperature', 'index_question'])['answer']
    .mean()
    .reset_index()
)

mean_per_question['question_num'] = (
    mean_per_question['index_question']
    .str.extract(r'(\d+)')
    .astype(int)
)

mean_per_question = mean_per_question.sort_values('question_num')

# =========================
# Paleta semântica por atividade
# =========================
activities = mean_per_question['id_activity'].unique()
base_cmap = plt.cm.tab10

activity_color_map = {
    activity: base_cmap(i % 10)
    for i, activity in enumerate(activities)
}

# =========================
# Gráfico único
# =========================
fig, ax = plt.subplots(figsize=(13, 8))

for (activity, model, temp), group in mean_per_question.groupby(
    ['id_activity', 'model', 'temperature']
):

    # Atividades extraídas
    if model == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns":
        label = f'{activity} | GPT-4.1 Treinado | Temp: {temp}'
        color = activity_color_map[activity]
        linewidth = 2.8
        alpha = 0.65
    else:
        label = 'Atividades extraídas'
        color = RED_CARO
        linewidth = 3.2
        alpha = 1.0  # sem transparência

    # Estilo de linha por temperatura
    if temp == '0.5':
        linestyle = '-'
    elif temp == '0.7':
        linestyle = ':'
    else:  # N/A
        linestyle = '--'

    ax.plot(
        group['question_num'],
        group['answer'],
        marker='o',
        linewidth=linewidth,
        linestyle=linestyle,
        color=color,
        alpha=alpha,
        label=label
    )

# =========================
# Eixos e layout
# =========================
ax.set_xticks(range(1, 11))
ax.set_xticklabels([f'P{i}' for i in range(1, 11)])

ax.set_ylim(1, 5.2)
ax.set_ylabel('Média das respostas (Likert)')
ax.set_xlabel('Pergunta')

ax.set_title(
    'Média das avaliações Likertpor pergunta\n'
    '(agrupado por atividade, modelo e temperatura) ',
    fontsize=13
)

ax.grid(axis='y', linestyle='--', alpha=0.4)

# =========================
# Legenda abaixo do gráfico
# =========================
handles, labels = ax.get_legend_handles_labels()

ax.legend(
    handles,
    labels,
    title='Configuração experimental',
    loc='upper center',
    bbox_to_anchor=(0.5, -0.28),
    ncol=2,
    frameon=False
)

plt.tight_layout()
plt.savefig(f'./artifacts/mean_likert_per_adicao_activities.png', dpi=300)
plt.show()

In [ ]:
# # =========================
# # Preparação dos dados
# # =========================
# df = final_df.copy()

# df = df[df['id_activity'] == "adição"]

# df.loc[df['model'] == 'N/A', 'temperature'] = 'N/A'
# df['temperature'] = df['temperature'].astype(str)

# df['source'] = df['model'].apply(
#     lambda x: 'Modelo Treinado' if x == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns"
#     else 'Atividades extraídas'
# )

# df['question_num'] = df['index_question'].str.extract('(\d+)').astype(int)

# df['config'] = (
#     df['id_activity'] + ' | ' +
#     df['source'] + ' | T=' + df['temperature']
# )

# # =========================
# # Paleta sóbria por assunto (CORRIGIDA)
# # =========================
# activities = df['id_activity'].unique()

# base_colors = sns.color_palette("muted", n_colors=len(activities))
# base_colors = [sns.desaturate(c, 0.40) for c in base_colors]
# base_colors = [tuple(np.clip(np.array(c) * 1.2, 0, 1)) for c in base_colors]

# activity_color_map = {
#     act: base_colors[i]
#     for i, act in enumerate(activities)
# }

# def vary_lightness(rgb, factors):
#     return [tuple(np.clip(np.array(rgb) * f, 0, 1)) for f in factors]

# palette = {}

# for act in activities:
#     configs_act = [c for c in df['config'].unique() if c.startswith(act + ' | ')]

#     # variações muito sutis de luminosidade
#     factors = np.linspace(0.85, 1.15, len(configs_act))
#     variations = vary_lightness(activity_color_map[act], factors)

#     for c, v in zip(configs_act, variations):
#         _, source, _ = c.split(' | ')
#         if source == 'Atividades extraídas':
#             palette[c] = RED_CARO
#         else:
#             palette[c] = v

# # =========================
# # Gráfico Boxplot
# # =========================
# plt.figure(figsize=(15, 8))

# def hue_sort_key(label):
#     act, source, temp = label.split(' | ')
    
#     if source == 'Atividades extraídas':
#         return (0, 0)
#     elif 'T=0.5' in temp:
#         return (1, 0)
#     elif 'T=0.7' in temp:
#         return (2, 0)
#     else:
#         return (3, 0)

# hue_order = sorted(df['config'].unique(), key=hue_sort_key)

# ax = sns.boxplot(
#     data=df,
#     x='question_num',
#     y='answer',
#     hue='config',
#     palette=palette,
#     showfliers=True,
#     hue_order=hue_order,
# )

# plt.ylim(0.8, 5.2)
# plt.xlabel('Pergunta')
# plt.ylabel('Distribuição das respostas (Likert)')
# plt.title('Distribuição das respostas Likert por pergunta,\nAdição, modelo e temperatura')

# plt.grid(axis='y', linestyle='--', alpha=0.35)

# # =========================
# # Rótulos do eixo X: P1...Pn
# # =========================
# current_labels = [t.get_text() for t in ax.get_xticklabels()]
# new_labels = [f'P{label}' for label in current_labels]
# ax.set_xticklabels(new_labels)

# # =========================
# # Legenda limpa
# # =========================
# handles, labels_leg = ax.get_legend_handles_labels()

# new_labels = []
# for label in labels_leg:
#     act, source, temp = label.split(' | ')
#     if source == 'Atividades extraídas':
#         new_labels.append('Atividades extraídas')
#     else:
#         new_labels.append(f'{act} | {source} | {temp}')

# ax.legend(
#     handles,
#     new_labels,
#     title='Configuração experimental',
#     loc='upper center',
#     bbox_to_anchor=(0.5, -0.25),
#     ncol=3,
#     frameon=False
# )

# plt.tight_layout()
# plt.savefig('./artifacts/boxplot_likert_por_assunto_modelo_temperatura_adicao.png', dpi=300)
# plt.show()

# =========================
# Data preparation
# =========================
df = final_df.copy()

# Keep Portuguese for filtering
df = df[df['id_activity'] == "adição"]

df.loc[df['model'] == 'N/A', 'temperature'] = 'N/A'
df['temperature'] = df['temperature'].astype(str)

df['source'] = df['model'].apply(
    lambda x: 'Trained Model' if x == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns"
    else 'Extracted Activities'
)

df['question_num'] = df['index_question'].str.extract('(\d+)').astype(int)

df['config'] = (
    df['id_activity'] + ' | ' +
    df['source'] + ' | T=' + df['temperature']
)

# =========================
# Translation (display only)
# =========================
activity_translation = {
    "adição": "Addition",
    "subtração": "Subtraction",
    "multiplicação": "Multiplication",
    "divisão": "Division"
}

activity_en = activity_translation.get("adição", "adição")

# =========================
# Muted palette per activity
# =========================
activities = df['id_activity'].unique()

base_colors = sns.color_palette("muted", n_colors=len(activities))
base_colors = [sns.desaturate(c, 0.40) for c in base_colors]
base_colors = [tuple(np.clip(np.array(c) * 1.2, 0, 1)) for c in base_colors]

activity_color_map = {
    act: base_colors[i]
    for i, act in enumerate(activities)
}

def vary_lightness(rgb, factors):
    return [tuple(np.clip(np.array(rgb) * f, 0, 1)) for f in factors]

palette = {}

for act in activities:
    configs_act = [c for c in df['config'].unique() if c.startswith(act + ' | ')]

    factors = np.linspace(0.85, 1.15, len(configs_act))
    variations = vary_lightness(activity_color_map[act], factors)

    for c, v in zip(configs_act, variations):
        _, source, _ = c.split(' | ')
        if source == 'Extracted Activities':
            palette[c] = RED_CARO
        else:
            palette[c] = v

# =========================
# Boxplot
# =========================
plt.figure(figsize=(15, 8))

def hue_sort_key(label):
    act, source, temp = label.split(' | ')
    
    if source == 'Extracted Activities':
        return (0, 0)
    elif 'T=0.5' in temp:
        return (1, 0)
    elif 'T=0.7' in temp:
        return (2, 0)
    else:
        return (3, 0)

hue_order = sorted(df['config'].unique(), key=hue_sort_key)

ax = sns.boxplot(
    data=df,
    x='question_num',
    y='answer',
    hue='config',
    palette=palette,
    showfliers=True,
    hue_order=hue_order,
)

plt.ylim(0.8, 5.2)
plt.xlabel('Question')
plt.ylabel('Response Distribution (Likert)')
plt.title(f'Likert Response Distribution by Question,\n{activity_en}, Model, and Temperature')

plt.grid(axis='y', linestyle='--', alpha=0.35)

# =========================
# X-axis labels: P1...Pn
# =========================
current_labels = [t.get_text() for t in ax.get_xticklabels()]
new_labels = [f'P{label}' for label in current_labels]
ax.set_xticklabels(new_labels)

# =========================
# Clean legend (translate activity only for display)
# =========================
handles, labels_leg = ax.get_legend_handles_labels()

new_labels = []
for label in labels_leg:
    act, source, temp = label.split(' | ')
    act_display = activity_translation.get(act, act)
    if source == 'Extracted Activities':
        new_labels.append('Extracted Activities')
    else:
        new_labels.append(f'{act_display} | {source} | {temp}')

ax.legend(
    handles,
    new_labels,
    title='Experimental Configuration',
    loc='upper center',
    bbox_to_anchor=(0.5, -0.25),
    ncol=3,
    frameon=False
)

plt.tight_layout()
plt.savefig('./artifacts/boxplot_likert_by_activity_model_temperature_addition.png', dpi=300)
plt.show()


## Print some questions to be displayed

### Adition - Extracted

In [ ]:
final_df[(final_df["index_question"] == "P7") & (final_df["id_activity"] == "adição") & (final_df["model"] == "N/A") & (final_df["answer"] == 5)]

In [ ]:
final_df[(final_df["index_question"] == "P7") & (final_df["id_activity"] == "adição") & (final_df["model"] == "N/A") & (final_df["answer"] == 1)]

### Addition - Model

In [ ]:
final_df[(final_df["index_question"] == "P7") & (final_df["id_activity"] == "adição") & (final_df["model"] == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns") & (final_df["answer"] == 5)]

In [ ]:
final_df[(final_df["index_question"] == "P7") & (final_df["id_activity"] == "adição") & (final_df["model"] == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns") & (final_df["answer"] == 1)]

### Subtraction

In [ ]:
final_df[(final_df["index_question"] == "P7") & (final_df["id_activity"] == "subtração") & (final_df["model"] == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns") & (final_df["answer"] == 5)]

In [ ]:
final_df[(final_df["index_question"] == "P7") & (final_df["id_activity"] == "subtração") & (final_df["model"] == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns") & (final_df["answer"] == 1)]

### Multiplication

In [ ]:
final_df[(final_df["index_question"] == "P7") & (final_df["id_activity"] == "multiplicação") & (final_df["model"] == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns") & (final_df["answer"] == 5)]

In [ ]:
final_df[(final_df["index_question"] == "P7") & (final_df["id_activity"] == "multiplicação") & (final_df["model"] == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns") & (final_df["answer"] == 1)]

### Divisão

In [ ]:
final_df[(final_df["index_question"] == "P7") & (final_df["id_activity"] == "divisão") & (final_df["model"] == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns") & (final_df["answer"] == 5)]

In [ ]:
final_df[(final_df["index_question"] == "P7") & (final_df["id_activity"] == "divisão") & (final_df["model"] == "ft:gpt-4.1-2025-04-14:ufcg::CikEQZns") & (final_df["answer"] == 1)]